In [11]:
pip install cryptography pillow


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import os
import json
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from PIL import Image
import io
import base64

In [13]:
# For academic testing, we use a static 32-byte AES key
MASTER_KEY = b"\x01" * 32  # DO NOT USE IN PROD


In [3]:
# ============================================================
# 1. Read image as bytes
# ============================================================
def load_image_bytes(path):
    with open(path, "rb") as f:
        return f.read()

In [4]:
# ============================================================
# 2. Generate CEK (AES-256-GCM)
# ============================================================
def generate_cek():
    return os.urandom(32)  # 256-bit CEK

In [5]:
# ============================================================
# 3. Encrypt image using CEK (AES-GCM)
# ============================================================
def encrypt_image_with_cek(image_bytes, cek):
    aesgcm = AESGCM(cek)
    nonce = os.urandom(12)

    ciphertext = aesgcm.encrypt(nonce, image_bytes, None)
    # AESGCM format = ciphertext + tag (tag = last 16 bytes)
    tag = ciphertext[-16:]
    encrypted_data = ciphertext[:-16]

    return encrypted_data, nonce, tag

In [6]:
# ============================================================
# 4. Wrap CEK using MASTER KEY (AES-GCM)
# ============================================================
def wrap_cek(cek, master_key):
    aesgcm = AESGCM(master_key)
    nonce = os.urandom(12)

    wrapped = aesgcm.encrypt(nonce, cek, None)
    tag = wrapped[-16:]
    encrypted_cek = wrapped[:-16]

    return encrypted_cek, nonce, tag


In [7]:
# ============================================================
# 5. Build final JSON secure object
# ============================================================
def create_encrypted_package(img_ct, img_nonce, img_tag, wrapped_cek, cek_nonce, cek_tag):
    package = {
        "ciphertext": base64.b64encode(img_ct).decode(),
        "nonce": base64.b64encode(img_nonce).decode(),
        "tag": base64.b64encode(img_tag).decode(),

        "wrapped_cek": base64.b64encode(wrapped_cek).decode(),
        "cek_nonce": base64.b64encode(cek_nonce).decode(),
        "cek_tag": base64.b64encode(cek_tag).decode(),

        "metadata": {
            "algorithm": "AES-256-GCM",
            "cek_wrapping": "AES-256-GCM",
            "version": 1
        }
    }
    return package


In [8]:
# ============================================================
# MAIN PIPELINE
# ============================================================
def envelope_encrypt_image(image_path, output_json):
    # 1. Load raw image bytes
    img_bytes = load_image_bytes(image_path)

    # 2. Generate CEK
    cek = generate_cek()

    # 3. Encrypt image with CEK
    img_ct, img_nonce, img_tag = encrypt_image_with_cek(img_bytes, cek)

    # 4. Wrap CEK using MASTER KEY
    wrapped_cek, cek_nonce, cek_tag = wrap_cek(cek, MASTER_KEY)

    # 5. Create JSON structure
    package = create_encrypted_package(
        img_ct, img_nonce, img_tag,
        wrapped_cek, cek_nonce, cek_tag
    )

    # Save JSON
    with open(output_json, "w") as f:
        json.dump(package, f, indent=4)

    print("Encrypted package created:", output_json)

In [14]:
# ============================================================
# Execute Example
# ============================================================
if __name__ == "__main__":
    envelope_encrypt_image("sample_img.jpg", "encrypted_output.json")

Encrypted package created: encrypted_output.json
